### f-строки

In [ ]:
import timeit

name = "Viktor"
age = 30

# Способ 1: Старый стиль с %
def format_percent():
    return "Имя: %s, Возраст: %s" % (name, age)

# Способ 2: Метод .format()
def format_dot_format():
    return "Имя: {}, Возраст: {}".format(name, age)

# Способ 3: F-строка (f-string)
def format_f_string():
    return f"Имя: {name}, Возраст: {age}"

# Способ 4: Конкатенация (для сравнения)
def format_concat():
    return "Имя: " + name + ", Возраст: " + str(age)

In [5]:
print("Стиль %:", timeit.timeit(format_percent, number=1000000))
print(".format():", timeit.timeit(format_dot_format, number=1000000))
print("F-строка:", timeit.timeit(format_f_string, number=1000000))
print("Конкатенация:", timeit.timeit(format_concat, number=1000000))

Стиль %: 0.1124650840065442
.format(): 0.1641010000021197
F-строка: 0.09776595799485222
Конкатенация: 0.1435094170155935


In [6]:
import dis

def format_dot_format():
    name = "Viktor"
    age = 30
    return "Имя: {}, Возраст: {}".format(name, age)

dis.dis(format_dot_format)

  3           0 RESUME                   0

  4           2 LOAD_CONST               1 ('Viktor')
              4 STORE_FAST               0 (name)

  5           6 LOAD_CONST               2 (30)
              8 STORE_FAST               1 (age)

  6          10 LOAD_CONST               3 ('Имя: {}, Возраст: {}')
             12 LOAD_ATTR                1 (NULL|self + format)
             32 LOAD_FAST                0 (name)
             34 LOAD_FAST                1 (age)
             36 CALL                     2
             44 RETURN_VALUE


In [7]:
def format_f_string():
    name = "Viktor"
    age = 30
    return f"Имя: {name}, Возраст: {age}"

dis.dis(format_f_string)

  1           0 RESUME                   0

  2           2 LOAD_CONST               1 ('Viktor')
              4 STORE_FAST               0 (name)

  3           6 LOAD_CONST               2 (30)
              8 STORE_FAST               1 (age)

  4          10 LOAD_CONST               3 ('Имя: ')
             12 LOAD_FAST                0 (name)
             14 FORMAT_VALUE             0
             16 LOAD_CONST               4 (', Возраст: ')
             18 LOAD_FAST                1 (age)
             20 FORMAT_VALUE             0
             22 BUILD_STRING             4
             24 RETURN_VALUE


In [8]:
# "Самодокументирующийся" дебаггинг
my_variable = 42
print(f"{my_variable=}")

my_variable=42


In [9]:
# Форматирование внутри
price = 59.99123
print(f"Цена: {price:.2f} рублей")

Цена: 59.99 рублей


In [10]:
# Отладочный вывод
text = "hello\nworld"
print(f"Обычный вывод: {text}")
print(f"Отладочный вывод: {text!r}")

Обычный вывод: hello
world
Отладочный вывод: 'hello\nworld'


### Копирование строк

In [11]:
original = "a_very_long_string_that_takes_up_some_memory" * 10

# Способ 1: Простое присваивание
copy1 = original

# Способ 2: Создание среза всей строки (классический способ скопировать последовательность)
copy2 = original[:]

print(id(original) == id(copy1))
print(id(original) == id(copy2))

True
True


### Короткие строки

In [12]:
a = "hello_world"
b = "hello_world"
print(a is b)

x = "hello world!"
y = "hello world!"
print(x is y) 

True
False


### Конкатенация строк

```python
def add_one_char(s):
    s = s + 'a'
    return s
```

- Временная сложность O(n)?
- Какой есть еще способ конкатенации символов в одну строку?

### Анализ производительности

#### 1. Анализ конкатенации в цикле (`s += char`)

**Стоимость отдельной операции:**

Стоимость k-ой операции сложения строк, где мы добавляем один символ к строке длиной `k-1`, пропорциональна новому размеру строки, то есть `k`.
Обозначим стоимость k-ой итерации как $C(k)$.

$$ C(k) \approx k $$

**Общая работа:**

Чтобы найти общую работу $W$ для создания строки длиной $N$, мы должны просуммировать стоимость всех операций от 1 до $N$.

$$ W = \sum_{k=1}^{N} C(k) \approx \sum_{k=1}^{N} k $$

**Вычисление суммы:**

Сумма первых $N$ натуральных чисел является классической арифметической прогрессией:

$$ \sum_{k=1}^{N} k = 1 + 2 + 3 + \dots + N = \frac{N(N+1)}{2} $$

**Анализ сложности (Big O):**

Раскроем скобки в формуле суммы, чтобы увидеть доминирующий член:

$$ \frac{N(N+1)}{2} = \frac{N^2 + N}{2} = \frac{1}{2}N^2 + \frac{1}{2}N $$

В нотации "Большого О" ($O$) мы отбрасываем константы ($\frac{1}{2}$) и члены меньшего порядка ($\frac{1}{2}N$), так как при больших $N$ именно $N^2$ определяет скорость роста.

$$ O\left(\frac{1}{2}N^2 + \frac{1}{2}N\right) \rightarrow O(N^2) $$

Таким образом, алгоритмическая сложность конкатенации строк в цикле является **квадратичной**.

---

#### 2. Анализ метода `"".join()`

Алгоритм `join` состоит из двух последовательных проходов.

**Проход 1: Расчет финального размера**

Алгоритм проходит по всем $M$ строкам в списке, чтобы посчитать их суммарную длину $N$. Сложность этого этапа пропорциональна суммарной длине всех строк.

$$ W_1 = O(N) $$

**Проход 2: Копирование данных**

Алгоритм выделяет один блок памяти размером $N$ и копирует в него все символы. Каждый символ копируется ровно один раз. Сложность этого этапа также пропорциональна суммарной длине.

$$ W_2 = O(N) $$

**Общая работа:**

Общая сложность — это сумма сложностей последовательных этапов.

$$ W_{total} = W_1 + W_2 = O(N) + O(N) = O(N) $$

Таким образом, алгоритмическая сложность метода `join` является **линейной**.

### Дизассемблирование

In [13]:
import dis
def add_one_char(s):
    s = s + 'a'
    return s
dis.dis(add_one_char)

  2           0 RESUME                   0

  3           2 LOAD_FAST                0 (s)
              4 LOAD_CONST               1 ('a')
              6 BINARY_OP                0 (+)
             10 STORE_FAST               0 (s)

  4          12 LOAD_FAST                0 (s)
             14 RETURN_VALUE


In [14]:
def build_string(n):
    s = ''
    for i in range(n):
        s = s + 'x'
    return s
dis.dis(build_string)

  1           0 RESUME                   0

  2           2 LOAD_CONST               1 ('')
              4 STORE_FAST               1 (s)

  3           6 LOAD_GLOBAL              1 (NULL + range)
             16 LOAD_FAST                0 (n)
             18 CALL                     1
             26 GET_ITER
        >>   28 FOR_ITER                 7 (to 46)
             32 STORE_FAST               2 (i)

  4          34 LOAD_FAST                1 (s)
             36 LOAD_CONST               2 ('x')
             38 BINARY_OP                0 (+)
             42 STORE_FAST               1 (s)
             44 JUMP_BACKWARD            9 (to 28)

  3     >>   46 END_FOR

  5          48 LOAD_FAST                1 (s)
             50 RETURN_VALUE


In [15]:
def build_string_join(n):
    char_list = []
    for i in range(n):
        char_list.append('x')
    return "".join(char_list)

dis.dis(build_string_join)

  1           0 RESUME                   0

  2           2 BUILD_LIST               0
              4 STORE_FAST               1 (char_list)

  3           6 LOAD_GLOBAL              1 (NULL + range)
             16 LOAD_FAST                0 (n)
             18 CALL                     1
             26 GET_ITER
        >>   28 FOR_ITER                19 (to 70)
             32 STORE_FAST               2 (i)

  4          34 LOAD_FAST                1 (char_list)
             36 LOAD_ATTR                3 (NULL|self + append)
             56 LOAD_CONST               1 ('x')
             58 CALL                     1
             66 POP_TOP
             68 JUMP_BACKWARD           21 (to 28)

  3     >>   70 END_FOR

  5          72 LOAD_CONST               2 ('')
             74 LOAD_ATTR                5 (NULL|self + join)
             94 LOAD_FAST                1 (char_list)
             96 CALL                     1
            104 RETURN_VALUE


### Сборщик мусора

In [16]:
import ctypes
import gc

class MyObject:
    pass

In [17]:
obj1 = MyObject()
obj2 = MyObject()
obj1.ref = obj2
obj2.ref = obj1

id1 = id(obj1)
id2 = id(obj2)
print(f"Объекты созданы с ID: {id1}, {id2}")

Объекты созданы с ID: 4719541232, 4719806800


In [18]:
del obj1
del obj2
print("Имена obj1 и obj2 удалены. Подсчет ссылок не справился...")

Имена obj1 и obj2 удалены. Подсчет ссылок не справился...


In [19]:
# Пытаемся "прочитать" имя класса по адресу в памяти
# Если объект жив, мы получим "MyObject"
print(f"По адресу {id1} находится: {ctypes.cast(id1, ctypes.py_object).value.__class__.__name__}")
print(f"По адресу {id2} находится: {ctypes.cast(id2, ctypes.py_object).value.__class__.__name__}")

По адресу 4719541232 находится: MyObject
По адресу 4719806800 находится: MyObject


In [20]:
collected_count = gc.collect() # Принудительный запуск циклического GC
print(f"\nСборщик мусора запущен. Собрано объектов: {collected_count}")


Сборщик мусора запущен. Собрано объектов: 2


In [ ]:
print("Пытаемся снова получить доступ к объектам...")
# Этот код вызовет Segmentation Fault или другую ошибку,
# так как мы пытаемся получить доступ к освобожденной памяти.
# Закомментируйте его перед запуском или будьте готовы к падению интерпретатора.
print(ctypes.cast(id1, ctypes.py_object).value)

: 

### Алгоритм запуска сборки

**1. Проверка Поколения 0 ($G_0$)**

Сборка мусора для поколения $G_0$ запускается, когда количество "чистых" созданий объектов в этом поколении превышает порог $t_0$.

Обозначим:
- $allocs_0$ — количество созданных объектов в $G_0$.
- $frees_0$ — количество объектов, уничтоженных в $G_0$ быстрым **подсчетом ссылок**.

Сборка для $G_0$ запускается, если:
$$ (allocs_0 - frees_0) > t_0 $$

По умолчанию $t_0 = 700$.

**После сборки $G_0$:**
- Счетчики $allocs_0$ и $frees_0$ сбрасываются в 0.
- Счетчик коллекций для следующего поколения ($k_1$) увеличивается на 1: $k_1 = k_1 + 1$.

**2. Проверка Поколения 1 ($G_1$)**

Сборка мусора для поколения $G_1$ (которая также затрагивает и $G_0$) запускается, когда счетчик коллекций $k_1$ достигает порога $t_1$.

Сборка для $G_1$ запускается, если:
$$ k_1 > t_1 $$

По умолчанию $t_1 = 10$.

**После сборки $G_1$:**
- Счетчик $k_1$ сбрасывается в 0.
- Счетчик коллекций для следующего поколения ($k_2$) увеличивается на 1: $k_2 = k_2 + 1$.

**3. Проверка Поколения 2 ($G_2$)**

Сборка мусора для поколения $G_2$ (полная сборка, затрагивающая все поколения) запускается, когда счетчик коллекций $k_2$ достигает порога $t_2$.

Сборка для $G_2$ запускается, если:
$$ k_2 > t_2 $$

По умолчанию $t_2 = 10$.

**После сборки $G_2$:**
- Счетчик $k_2$ сбрасывается в 0.

In [1]:
import gc
print(gc.get_threshold())

(700, 10, 10)


In [2]:
print(gc.get_count())

(200, 1, 5)
